### KJØR UTILS!!

In [1]:
import utils
import keras
import numpy as np
from aim.keras import AimCallback

2026-03-04 22:23:50.028834: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 22:23:50.106629: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 22:23:59.454910: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
(train_x, train_y), (val_x, val_y), (test_x, test_y) = utils.read_andre_data()

Correction complete!
 - Files found as-is: 2078
 - Extensions corrected: 955
 - Still missing (not found): 0
Correction complete!
 - Files found as-is: 437
 - Extensions corrected: 213
 - Still missing (not found): 0
Correction complete!
 - Files found as-is: 449
 - Extensions corrected: 202
 - Still missing (not found): 0


In [3]:
print(train_x.shape)
print(train_y.shape)
print(val_x.shape)
print(val_y.shape)
print(test_x.shape)
print(test_y.shape)


(3033, 300, 300, 3)
(3033, 10)
(650, 300, 300, 3)
(650, 10)
(651, 300, 300, 3)
(651, 10)


In [4]:
input = keras.Input(shape=(train_x[0].shape))

x = keras.layers.Conv2D(filters=32, kernel_size=3, activation="relu")(input)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Flatten()(x)

gate1 = keras.layers.Dense(1, activation="sigmoid", name="gate1")(x)
gate2 = keras.layers.Dense(8, activation="softmax", name="gate2")(x)

model = keras.Model(inputs=input, outputs={"gate1": gate1, "gate2": gate2}, name="Overfitted_model")
model.summary()

opt = keras.optimizers.Adam(learning_rate=6e-4)

model.compile(
    optimizer=opt,
    loss={
        "gate1": keras.losses.BinaryCrossentropy(),
        "gate2": keras.losses.SparseCategoricalCrossentropy(),
    },
    metrics={
        "gate1": [keras.metrics.BinaryAccuracy(name="acc")],
    },
    weighted_metrics={
        "gate2": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
    },
)


I0000 00:00:1772659498.772727   18968 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9569 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "Overfitted_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 298, 298,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 149, 149,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 73, 73,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 71, 71,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 35, 35,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 156800)    │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gate1 (Dense)       │ (None, 1)         │    156,801 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gate2 (Dense)       │ (None, 8)         │  1,254,408 │ flatten[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,504,457 (5.74 MB)

 Trainable params: 1,504,457 (5.74 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
batch_size = 16
epochs = 10
aim_cb = AimCallback(
    repo=".",                      
    experiment="Vehicle_Gate_Test",
)

hparams = {
    "learning_rate": opt.learning_rate,
    "batch_size": batch_size,
    "epochs": epochs,
    "optimizer": "Adam",
    "model": "Overfitted_model",
}

sample_weight={
    "gate1": np.ones(len(train_y)),
    "gate2": 1 - train_y['gate1'] 
}

validation_data=(val_x, {'gate1': val_y["gate1"], 'gate2': val_y["gate2"]})

y = {'gate1': train_y["gate1"], 'gate2': train_y["gate2"]}
history = model.fit(epochs=epochs, batch_size=batch_size, x=train_x, y=y, sample_weight=sample_weight, validation_data=validation_data, callbacks=[aim_cb])

Epoch 1/10


2026-03-04 22:25:08.405697: I external/local_xla/xla/service/service.cc:163] XLA service 0x7639b8006640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-04 22:25:08.405737: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3080, Compute Capability 8.6
2026-03-04 22:25:08.504037: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-04 22:25:08.823578: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


 10/190 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - gate1_acc: 0.4602 - gate1_loss: 0.8104 - gate2_acc: 0.7673 - gate2_loss: 0.4577 - loss: 1.2681

I0000 00:00:1772659513.221671   19346 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


190/190 ━━━━━━━━━━━━━━━━━━━━ 15s 46ms/step - gate1_acc: 0.6858 - gate1_loss: 0.5993 - gate2_acc: 0.9944 - gate2_loss: 0.0096 - loss: 0.6087 - val_gate1_acc: 0.7123 - val_gate1_loss: 0.5370 - val_gate2_acc: 0.4738 - val_gate2_loss: 6.0877 - val_loss: 6.6376
Epoch 2/10
190/190 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - gate1_acc: 0.7738 - gate1_loss: 0.4719 - gate2_acc: 1.0000 - gate2_loss: 3.8809e-05 - loss: 0.4722 - val_gate1_acc: 0.7754 - val_gate1_loss: 0.4718 - val_gate2_acc: 0.4738 - val_gate2_loss: 9.5390 - val_loss: 10.0286
Epoch 3/10
190/190 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - gate1_acc: 0.8378 - gate1_loss: 0.3685 - gate2_acc: 1.0000 - gate2_loss: 4.5062e-06 - loss: 0.3691 - val_gate1_acc: 0.8015 - val_gate1_loss: 0.4698 - val_gate2_acc: 0.4738 - val_gate2_loss: 14.2162 - val_loss: 14.7204
Epoch 4/10
190/190 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - gate1_acc: 0.8952 - gate1_loss: 0.2487 - gate2_acc: 1.0000 - gate2_loss: 1.2320e-07 - loss: 0.2484 - val_gate1_acc: 0.7677 - val_gate1_loss: 0.

In [11]:
def predict_heads(model, data_x):
    """Returns (lvl1_probs, lvl2_probs) for the overfitted model."""
    preds = model.predict(data_x, verbose=0)
    # Mapping 'gate1' -> lvl1 and 'gate2' -> lvl2
    return preds['gate1'].flatten(), preds['gate2']

# Get predictions once to use in all tables
p1, p2 = predict_heads(model, val_x)
# Extract ground truth as numpy arrays
y1_true = val_y['gate1'].to_numpy()
y2_true = val_y['gate2'].to_numpy()

In [13]:
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score

def eval_lvl1_by_lighting(y_true, p1_probs, lighting_series, threshold=0.5):
    rows = []
    y_pred = (p1_probs >= threshold).astype(int)
    
    for cond in ["Light", "Medium", "Dark"]:
        mask = (lighting_series == cond)
        if not mask.any(): continue
        
        yt, yp = y_true[mask], y_pred[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        
        rows.append({
            "lighting": cond,
            "n_total": len(yt),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp,
            "TPR": tp / (tp + fn) if (tp + fn) > 0 else 0,
            "TNR": tn / (tn + fp) if (tn + fp) > 0 else 0,
            "lvl1_acc": accuracy_score(yt, yp),
            "lvl1_bal_acc": balanced_accuracy_score(yt, yp),
            "lvl1_f1": f1_score(yt, yp)
        })
    return pd.DataFrame(rows)

print("\n--- TABLE 1: LEVEL 1 BY LIGHTING ---")
display(eval_lvl1_by_lighting(y1_true, p1, val_y['lighting']))


--- TABLE 1: LEVEL 1 BY LIGHTING ---


,lighting,n_total,TN,FP,FN,TP,TPR,TNR,lvl1_acc,lvl1_bal_acc,lvl1_f1
0,Light,441,227,54,43,117,0.731250,0.807829,0.780045,0.769540,0.706949
1,Medium,104,5,8,11,80,0.879121,0.384615,0.817308,0.631868,0.893855
2,Dark,105,6,8,9,82,0.901099,0.428571,0.838095,0.664835,0.906077


In [22]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support
import pandas as pd
# 1. Get Predictions
preds = model.predict(val_x, verbose=0)
p1 = preds['gate1'].flatten() # Level 1 probabilities
p2 = preds['gate2']           # Level 2 probabilities (softmax)

# 2. Prepare Ground Truth & Predictions
y1_true = val_y['gate1'].to_numpy()
y2_true = val_y['gate2'].to_numpy()
y2_pred = np.argmax(p2, axis=1)

# Mask for Tesla samples only (Level 2 is only valid when Lvl1 == 1)
tesla_mask = (y1_true == 1)
yt2_tesla = y2_true[tesla_mask]
yp2_tesla = y2_pred[tesla_mask]
lighting_tesla = val_y['lighting'].to_numpy()[tesla_mask]

# --- TABLE: LEVEL 2 OVERALL ---
def eval_lvl2_overall(yt, yp):
    return pd.DataFrame([{
        "n_tesla": len(yt),
        "lvl2_acc": accuracy_score(yt, yp),
        "lvl2_bal_acc": balanced_accuracy_score(yt, yp),
        "lvl2_f1_macro": f1_score(yt, yp, average='macro'),
        "lvl2_maj_acc": (yt == pd.Series(yt).mode()[0]).mean()
    }])

print("\n--- LEVEL 2 OVERALL ---")
display(eval_lvl2_overall(yt2_tesla, yp2_tesla))

# --- TABLE: LEVEL 2 BY LIGHTING ---
def eval_lvl2_lighting(yt, yp, light):
    rows = []
    for cond in ["Light", "Medium", "Dark"]:
        m = (light == cond)
        if not m.any(): continue
        rows.append({
            "lighting": cond,
            "n_tesla": m.sum(),
            "lvl2_acc": accuracy_score(yt[m], yp[m]),
            "lvl2_bal_acc": balanced_accuracy_score(yt[m], yp[m]),
            "lvl2_f1_macro": f1_score(yt[m], yp[m], average='macro')
        })
    return pd.DataFrame(rows)

print("\n--- LEVEL 2 BY LIGHTING ---")
display(eval_lvl2_lighting(yt2_tesla, yp2_tesla, lighting_tesla))

# --- TABLE: LEVEL 2 PER-CLASS ---
p, r, f, s = precision_recall_fscore_support(yt2_tesla, yp2_tesla, labels=range(len(TESLA_NAMES)), zero_division=0)
class_metrics = pd.DataFrame({
    'class_id': range(len(TESLA_NAMES)),
    'class': TESLA_NAMES,
    'support': s,
    'precision': p,
    'recall': r,
    'f1': f
}).sort_values('recall') # Baseline sorts by recall

print("\n--- LEVEL 2 PER-CLASS ---")
display(class_metrics)

# --- TABLE: TOP MISCLASSIFICATIONS ---
misclassed = []
for t_id in range(len(TESLA_NAMES)):
    for p_id in range(len(TESLA_NAMES)):
        if t_id == p_id: continue
        count = ((yt2_tesla == t_id) & (yp2_tesla == p_id)).sum()
        support = (yt2_tesla == t_id).sum()
        if count > 0:
            misclassed.append({
                'true': TESLA_NAMES[t_id],
                'pred': TESLA_NAMES[p_id],
                'rate': count / support,
                'support_true': support
            })

top_mis = pd.DataFrame(misclassed).sort_values('rate', ascending=False).head(5)
print("\n--- TOP MISCLASSIFICATIONS ---")
display(top_mis)


--- LEVEL 2 OVERALL ---


/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,n_tesla,lvl2_acc,lvl2_bal_acc,lvl2_f1_macro,lvl2_maj_acc
0,342,0.0,0.0,0.0,0.374269



--- LEVEL 2 BY LIGHTING ---


/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,lighting,n_tesla,lvl2_acc,lvl2_bal_acc,lvl2_f1_macro
0,Light,160,0.0,0.0,0.0
1,Medium,91,0.0,0.0,0.0
2,Dark,91,0.0,0.0,0.0



--- LEVEL 2 PER-CLASS ---


,class_id,class,support,precision,recall,f1
0,0,3 2017–2023,52.0,0.0,0.0,0.0
1,1,3 2024–nå,37.0,0.0,0.0,0.0
2,2,S 2012–2015,16.0,0.0,0.0,0.0
3,3,S 2016–nå,19.0,0.0,0.0,0.0
4,4,X,41.0,0.0,0.0,0.0
5,5,Y 2020–2024,128.0,0.0,0.0,0.0
6,6,Y 2025-nå,49.0,0.0,0.0,0.0


KeyError: 'rate'